Samantha Asefi (sma9am@virginia.edu)
DS 5001
8 May 2026

# Bag of Words, TFIDF, DTCM

## imports

In [1]:
import os
import re
from glob import glob
from pathlib import Path
%pip install kaleido
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
import matplotlib.pyplot as plt

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('tagsets')

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!


True

## OHCO and Bag Levels Defined

In this section, there is a clear bag being defined - CHAPS. This is controlling how the granularity breakdown of our data looks. The goal of all of these in the rest of this notebook is to view how the count is used in each of the chapters. 

Important choices are made here: 
- count_method: this choice was to use 'n' which means that the number of times the word is used is documented
- bag = CHAPS: The bag being chaps is important because there are essentially 63 documents in this corpus at the chapter level, so it gives just the right amount of zoom in. 

In [2]:
count_method = 'n' # 'c' or 'n' # n = n tokens, c = distinct token (term) count
tf_method = 'sum' # sum, max, log, double_norm, raw, binary
tf_norm_k = .5 # only used for double_norm
idf_method = 'standard' # standard, max, smooth
gradient_cmap = 'YlGnBu' # YlGn, GnBu, YlGnBu; For tables; see https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html 

OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
SENTS = OHCO[:4]
PARAS = OHCO[:3]
CHAPS = OHCO[:2]
BOOKS = OHCO[:1]

bag = CHAPS

In [3]:
data_dir = Path("corpus/f2")

LIB = pd.read_csv(data_dir / "LIB.csv").set_index(['book_id', 'chap_num'])
TOKEN = pd.read_csv(data_dir / "TOKEN.csv").set_index(OHCO)
VOCAB = pd.read_csv(data_dir / "VOCAB.csv").set_index('term_str')

### Adding term id as an index

In [4]:
TOKEN = TOKEN.reset_index()

TOKEN = TOKEN.merge(
    VOCAB.reset_index()[['term_id', 'term_str']],
    on='term_str',
    how='left'
)

TOKEN = TOKEN.set_index(OHCO).sort_index()

In [5]:
LIB = pd.read_csv(data_dir / "LIB.csv").set_index(BOOKS)

LIB.head()

,chap_num,author,book_title,filepath,source_url,page_title,chapter_title,n_paragraphs_raw
book_id,,,,,,,,
fourier_selections,1,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch01.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of the Role of the Passions,41
fourier_selections,2,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch02.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of Education,23
fourier_selections,3,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch03.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Universal Harmony”,10
fourier_selections,4,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch04.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Letter to the High Judge”,43
fourier_selections,5,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch05.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Indices and Methods which led to the Discovery”,34


In [6]:
TOKEN = pd.read_csv(data_dir / 'TOKEN.csv').set_index(OHCO)
TOKEN.head()

pos_tuple  \
book_id            chap_num para_num sent_num token_num                            
fourier_selections 1        3        0        2          ('philosophical', 'JJ')   
                                              3                 ('whims', 'NNS')   
                                              4                ('called', 'VBN')   
                                              5                ('duties', 'NNS')   
                                              8               ('relation', 'NN')   

                                                             token_str  pos  \
book_id            chap_num para_num sent_num token_num                       
fourier_selections 1        3        0        2          philosophical   JJ   
                                              3                  whims  NNS   
                                              4                 called  VBN   
                                              5                 duties  NNS   
                                              8               relation   NN   

                                                              term_str  \
book_id            chap_num para_num sent_num token_num                  
fourier_selections 1        3        0        2          philosophical   
                                              3                  whims   
                                              4                 called   
                                              5                 duties   
                                              8               relation   

                                                         is_stop  
book_id            chap_num para_num sent_num token_num           
fourier_selections 1        3        0        2            False  
                                              3            False  
                                              4            False  
                                              5            False  
                                              8            False

In [7]:
VOCAB = pd.read_csv(data_dir / 'VOCAB.csv').set_index('term_str')
VOCAB.head()

,term_id,n,pos_max,num,stop,p_stem
term_str,,,,,,
aback,0,34,NN,0,0,aback
abandon,1,36,VB,0,0,abandon
abandoned,2,34,VBN,0,0,abandon
abandoning,3,25,VBG,0,0,abandon
abandonment,4,3,NN,0,0,abandon


# Add Max POS to VOCAB

In [8]:
TOKEN.groupby(['term_str', 'pos']).pos.count()
TOKEN.groupby(['term_str', 'pos']).pos.count().unstack()
TOKEN.groupby(['term_str', 'pos']).pos.count().unstack().idxmax(1)

term_str
aback           NN
abandon         VB
abandoned      VBN
abandoning     VBG
abandonment     NN
              ... 
zeit           NNP
zeitung        NNP
zero            NN
zone            NN
zones          NNS
Length: 9667, dtype: str

In [9]:
VOCAB['pos_max'] = TOKEN.groupby(['term_str', 'pos']).pos.count().unstack().idxmax(1)
VOCAB.sample(5)

,term_id,n,pos_max,num,stop,p_stem
term_str,,,,,,
crowd,2025,2,NN,0,0,crowd
falling,3312,13,VBG,0,0,fall
cubs,2043,10,NN,0,0,cub
endure,2921,1,VB,0,0,endur
chasm,1292,3,NN,0,0,chasm


# Compare POS Stats in TOKEN and Vocab

This POS distribution chart shows that Nouns, Proper Nouns, and adjectives are the most popular POS in this corpus.

In [10]:
POS = TOKEN.pos.value_counts().to_frame(name='n')
POS.index.name = 'pos_id'

In [11]:
fig, ax = plt.subplots(figsize=(15, 5))
POS.sort_values('n').plot.bar(y='n', ax=ax, rot=45)
ax.set_title('POS Distribution')
plt.tight_layout()
plt.savefig('pos_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

# Zipf's Law
## Add term rank to VOCAB

In [12]:
if 'term_rank' not in VOCAB.columns:
    VOCAB = VOCAB.sort_values('n', ascending=False).reset_index()
    VOCAB.index.name = 'term_rank'
    VOCAB = VOCAB.reset_index()
    VOCAB = VOCAB.set_index('term_id')
    VOCAB['term_rank'] = VOCAB['term_rank'] + 1

VOCAB.head()

,term_rank,term_str,n,pos_max,num,stop,p_stem
term_id,,,,,,,
530,1,army,6934,NNP,0,0,armi
6108,2,party,4661,NNP,0,0,parti
6197,3,people,4016,NNS,0,0,peopl
1322,4,china,3832,NNP,0,0,china
7368,5,revolution,3697,NN,0,0,revolut


Now we can see the most popular words in the corpus and it looks like words that are heavily influenced by war and uprising. Politics etc.

## Alternate Rank

In [13]:
new_rank = VOCAB.n.value_counts()\
    .sort_index(ascending=False).reset_index().reset_index()\
    .rename(columns={'level_0':'term_rank2', 'index':'n', 'n':'nn'})\
    .set_index('n')

new_rank.head()

,nn,count
n,,
0,6934,1
1,4661,1
2,4016,1
3,3832,1
4,3697,1


In [14]:
VOCAB['term_rank2'] = VOCAB.n.rank(method='dense', ascending=False).astype(int)
VOCAB.head()

,term_rank,term_str,n,pos_max,num,stop,p_stem,term_rank2
term_id,,,,,,,,
530,1,army,6934,NNP,0,0,armi,1
6108,2,party,4661,NNP,0,0,parti,2
6197,3,people,4016,NNS,0,0,peopl,3
1322,4,china,3832,NNP,0,0,china,4
7368,5,revolution,3697,NN,0,0,revolut,5


In [15]:
VOCAB['p'] = VOCAB.n / VOCAB.shape[0]

## Compute Zipf's K

In [16]:
VOCAB['zipf_k'] = VOCAB.n * VOCAB.term_rank
VOCAB['zipf_k2'] = VOCAB.n * VOCAB.term_rank2
VOCAB['zipf_k3'] = VOCAB.p * VOCAB.term_rank2

In [17]:
VOCAB.describe().T

,count,mean,std,min,25%,50%,75%,max
term_rank,9668.0,4834.500000,2791.055535,1.000000,2417.750000,4834.50000,7251.250000,9668.000000
n,9668.0,45.211109,180.555800,1.000000,1.000000,3.00000,31.000000,6934.000000
num,9668.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
stop,9668.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
term_rank2,9668.0,463.504034,74.771621,1.000000,467.000000,495.00000,497.000000,497.000000
p,9668.0,0.004676,0.018676,0.000103,0.000103,0.00031,0.003206,0.717211
zipf_k,9668.0,35960.084506,33866.978940,6195.000000,8610.750000,14605.50000,71091.250000,100632.000000
zipf_k2,9668.0,10876.515619,16999.225195,497.000000,497.000000,1485.00000,14477.000000,67095.000000
zipf_k3,9668.0,1.125002,1.758298,0.051407,0.051407,0.15360,1.497414,6.939905


## Words with low K

These words that don't appear that much and are not important.

In [18]:
VOCAB[VOCAB.zipf_k <= VOCAB.zipf_k.quantile(.1)].sort_values('zipf_k3', ascending=True).head()

,term_rank,term_str,n,pos_max,num,stop,p_stem,term_rank2,p,zipf_k,zipf_k2,zipf_k3
term_id,,,,,,,,,,,,
3402,6677,festivals,1,NNS,0,0,festiv,497,0.000103,6677,497,0.051407
9065,6830,uneasiness,1,JJ,0,0,uneasi,497,0.000103,6830,497,0.051407
9066,6831,unemployed,1,JJ,0,0,unemploy,497,0.000103,6831,497,0.051407
214,6832,afflicting,1,VBG,0,0,afflict,497,0.000103,6832,497,0.051407
9068,6833,unendurable,1,JJ,0,0,unendur,497,0.000103,6833,497,0.051407


## Words with high k

Words with high K are usually those that seem to be noise

In [19]:
VOCAB[VOCAB.zipf_k >= VOCAB.zipf_k.quantile(.9)].sort_values('zipf_k3', ascending=False).head()

,term_rank,term_str,n,pos_max,num,stop,p_stem,term_rank2,p,zipf_k,zipf_k2,zipf_k3
term_id,,,,,,,,,,,,
1718,546,congress,168,NNP,0,0,congress,331,0.017377,91728,55608,5.751758
9657,551,yunnan,168,NNP,0,0,yunnan,331,0.017377,92568,55608,5.751758
4046,553,hirota,168,NNP,0,0,hirota,331,0.017377,92904,55608,5.751758
3310,552,fall,168,NN,0,0,fall,331,0.017377,92736,55608,5.751758
3650,547,fuhsien,168,NNP,0,0,fuhsien,331,0.017377,91896,55608,5.751758


# Visualize

In [20]:
px.histogram(VOCAB, 'zipf_k', marginal='box')

In [21]:
px.histogram(VOCAB, 'zipf_k2', marginal='box')

In [22]:
px.histogram(VOCAB, 'zipf_k3', marginal='box')

## Ranks and N

In [23]:
VSAMP1 = VOCAB.reset_index()[['n', 'term_rank', 'zipf_k', 'term_str', 'pos_max']]

In [24]:
px.scatter(VSAMP1, x='term_rank', y='n', log_y=False, log_x=False, hover_name='term_str', color='pos_max')

In [25]:
px.scatter(VSAMP1, x='term_rank', y='n', log_y=True, log_x=True, hover_name='term_str', color='pos_max')

## Demo Rank

In [26]:
rank_index = [1, 2, 3, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]

In [27]:
demo = VOCAB.loc[VOCAB.term_rank.isin(rank_index), ['term_str', 'term_rank', 'n', 'zipf_k', 'pos_max']]

In [28]:
demo.style.background_gradient(cmap=gradient_cmap, high=.5)

,term_str,term_rank,n,zipf_k,pos_max
term_id,,,,,
530,army,1,6934,6934,NNP
6108,party,2,4661,9322,NNP
6197,people,3,4016,12048,NNS
5623,national,10,2496,24960,JJ
9099,united,20,1757,35140,JJ
4793,kuomintang,30,1379,41370,NNP
4257,imperialists,40,1162,46480,NNS
5070,long,50,995,49750,JJ
1389,classes,60,815,48900,NNS


# Vocab Entropy

In [29]:
VOCAB['p2'] = VOCAB.n / VOCAB.n.sum()

In [30]:
VOCAB['h'] = VOCAB.p2 * np.log2(1/VOCAB.p2) # Self entropy of each word 
H = VOCAB.h.sum()
N_v = VOCAB.shape[0]
H_max = np.log2(N_v)
R = round(1 - (H/H_max), 2) * 100

In [31]:
print("H \t= {}\nH_max \t= {}\nR \t= {}%".format(H, H_max, int(R)))

H 	= 10.733215454768857
H_max 	= 13.239001757765582
R 	= 19%


In [32]:
TOK = TOKEN.reset_index().merge(
    VOCAB.reset_index()[['term_id', 'term_str']],
    on='term_str',
    how='left'
)

print(TOK[['term_str', 'term_id']].head())
print("Missing term_id:", TOK['term_id'].isna().sum())

        term_str  term_id
0  philosophical     6300
1          whims     9486
2         called     1119
3         duties     2716
4       relation     7120
Missing term_id: 0


# BOW

Bag of words is what described the occurences of words within a document and ignores the grammatical word order

In [33]:
BOW = (
    TOK
    .groupby(bag + ['term_id'])
    .size()
    .to_frame('n')
)

BOW['c'] = 1
BOW.head()

n  c
book_id            chap_num term_id      
fourier_selections 1        9        1  1
                            14       1  1
                            31       1  1
                            35       1  1
                            38       1  1

In [34]:
BOW['c'] = BOW.n.astype('bool').astype('int')

In [35]:
BOW.head(10)

n  c
book_id            chap_num term_id      
fourier_selections 1        9        1  1
                            14       1  1
                            31       1  1
                            35       1  1
                            38       1  1
                            73       1  1
                            74       1  1
                            75       1  1
                            79       2  1
                            105      1  1

# Document Term Matrix (DTM)

In [36]:
DTCM = BOW[count_method].unstack().fillna(0).astype('int')

In [37]:
DTCM = BOW[count_method].unstack().fillna(0).astype('int')
DTCM.head()

term_id                      0     1     2     3     4     5     6     7     \
book_id            chap_num                                                   
fourier_selections 1            0     0     0     0     0     0     0     0   
                   2            0     0     0     0     0     0     0     0   
                   3            0     0     0     0     0     0     0     0   
                   4            0     0     0     0     0     0     0     0   
                   5            0     0     0     0     0     0     1     2   

term_id                      8     9     ...  9658  9659  9660  9661  9662  \
book_id            chap_num              ...                                 
fourier_selections 1            0     1  ...     0     0     0     0     0   
                   2            0     0  ...     0     0     0     0     0   
                   3            0     0  ...     0     0     0     0     0   
                   4            0     0  ...     0     0     0     0     0   
                   5            0     0  ...     0     0     0     0     0   

term_id                      9663  9664  9665  9666  9667  
book_id            chap_num                                
fourier_selections 1            0     0     0     0     0  
                   2            0     0     0     0     0  
                   3            0     0     0     0     0  
                   4            0     0     0     0     0  
                   5            0     0     0     0     0  

[5 rows x 9668 columns]

In [38]:
print('TF method:', tf_method)

if tf_method == 'sum':
    TF = DTCM.T / DTCM.T.sum()

elif tf_method == 'max':
    TF = DTCM.T / DTCM.T.max()

elif tf_method == 'log':
    TF = np.log10(1 + DTCM.T)
    
elif tf_method == 'raw':
    TF = DTCM.T

elif tf_method == 'double_norm':
    TF = DTCM.T / DTCM.T.max()
    TF = tf_norm_k + (1 - tf_norm_k) * TF[TF > 0] # EXPLAIN; may defeat purpose of norming

elif tf_method == 'binary':
    TF = DTCM.T.astype('bool').astype('int')
    
TF = TF.T

TF method: sum


In [39]:
if 'TF' not in globals():
	print('TF not found — computing TF using tf_method:', tf_method)
	if tf_method == 'sum':
		TF = DTCM.T / DTCM.T.sum()
	elif tf_method == 'max':
		TF = DTCM.T / DTCM.T.max()
	elif tf_method == 'log':
		TF = np.log10(1 + DTCM.T)
	elif tf_method == 'raw':
		TF = DTCM.T
	elif tf_method == 'double_norm':
		TF = DTCM.T / DTCM.T.max()
		TF = tf_norm_k + (1 - tf_norm_k) * TF[TF > 0]
	elif tf_method == 'binary':
		TF = DTCM.T.astype('bool').astype('int')
	TF = TF.T
else:
	print('TF already defined — displaying head')
TF.head()

TF already defined — displaying head


term_id                      0     1     2     3     4     5         6     \
book_id            chap_num                                                 
fourier_selections 1          0.0   0.0   0.0   0.0   0.0   0.0  0.000000   
                   2          0.0   0.0   0.0   0.0   0.0   0.0  0.000000   
                   3          0.0   0.0   0.0   0.0   0.0   0.0  0.000000   
                   4          0.0   0.0   0.0   0.0   0.0   0.0  0.000000   
                   5          0.0   0.0   0.0   0.0   0.0   0.0  0.000634   

term_id                          7     8         9     ...  9658  9659  9660  \
book_id            chap_num                            ...                     
fourier_selections 1         0.000000   0.0  0.000657  ...   0.0   0.0   0.0   
                   2         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   3         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   4         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   5         0.001267   0.0  0.000000  ...   0.0   0.0   0.0   

term_id                      9661  9662  9663  9664  9665  9666  9667  
book_id            chap_num                                            
fourier_selections 1          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   2          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   3          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   4          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   5          0.0   0.0   0.0   0.0   0.0   0.0   0.0  

[5 rows x 9668 columns]

# Compute DF

In [40]:
DF = DTCM[DTCM > 0].count()
DF = DTCM[DTCM > 0].count()
DF.head()

term_id
0    1
1    3
2    5
3    3
4    3
dtype: int64

# Comput IDF

In [41]:
N = DTCM.shape[0]

print('IDF method:', idf_method)

if idf_method == 'standard':
    IDF = np.log10(N / DF)

elif idf_method == 'max':
    IDF = np.log10(DF.max() / DF) 

elif idf_method == 'smooth':
    IDF = np.log10((1 + N) / (1 + DF)) + 1 # Correct?

IDF method: standard


# Compute TF IDF

In [42]:
TFIDF = TF * IDF

In [43]:
TFIDF.head()

term_id                      0     1     2     3     4     5        6     \
book_id            chap_num                                                
fourier_selections 1          0.0   0.0   0.0   0.0   0.0   0.0  0.00000   
                   2          0.0   0.0   0.0   0.0   0.0   0.0  0.00000   
                   3          0.0   0.0   0.0   0.0   0.0   0.0  0.00000   
                   4          0.0   0.0   0.0   0.0   0.0   0.0  0.00000   
                   5          0.0   0.0   0.0   0.0   0.0   0.0  0.00114   

term_id                          7     8         9     ...  9658  9659  9660  \
book_id            chap_num                            ...                     
fourier_selections 1         0.000000   0.0  0.001181  ...   0.0   0.0   0.0   
                   2         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   3         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   4         0.000000   0.0  0.000000  ...   0.0   0.0   0.0   
                   5         0.002281   0.0  0.000000  ...   0.0   0.0   0.0   

term_id                      9661  9662  9663  9664  9665  9666  9667  
book_id            chap_num                                            
fourier_selections 1          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   2          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   3          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   4          0.0   0.0   0.0   0.0   0.0   0.0   0.0  
                   5          0.0   0.0   0.0   0.0   0.0   0.0   0.0  

[5 rows x 9668 columns]

In [44]:
VOCAB['df'] = DF
VOCAB['idf'] = IDF

In [45]:
VOCAB.head()

,term_rank,term_str,n,pos_max,num,stop,p_stem,term_rank2,p,zipf_k,zipf_k2,zipf_k3,p2,h,df,idf
term_id,,,,,,,,,,,,,,,,
530,1,army,6934,NNP,0,0,armi,1,0.717211,6934,6934,0.717211,0.015864,0.094835,18,0.544068
6108,2,party,4661,NNP,0,0,parti,2,0.482106,9322,9322,0.964212,0.010663,0.069858,21,0.477121
6197,3,people,4016,NNS,0,0,peopl,3,0.415391,12048,12048,1.246173,0.009188,0.062165,36,0.243038
1322,4,china,3832,NNP,0,0,china,4,0.396359,15328,15328,1.585436,0.008767,0.059910,11,0.757948
7368,5,revolution,3697,NN,0,0,revolut,5,0.382396,18485,18485,1.911978,0.008458,0.058237,22,0.456918


In [46]:
BOW['tf'] = TF.stack()
BOW['tfidf'] = TFIDF.stack()

In [47]:
BOW.head()

n  c        tf     tfidf
book_id            chap_num term_id                          
fourier_selections 1        9        1  1  0.000657  0.001181
                            14       1  1  0.000657  0.000287
                            31       1  1  0.000657  0.001181
                            35       1  1  0.000657  0.000868
                            38       1  1  0.000657  0.001181

In [48]:
VOCAB['tfidf_sum'] = TFIDF.sum()

In [49]:
VOCAB.sort_values('tfidf_sum', ascending=False).head(20).style.background_gradient(cmap=gradient_cmap, high=1)

,term_rank,term_str,n,pos_max,num,stop,p_stem,term_rank2,p,zipf_k,zipf_k2,zipf_k3,p2,h,df,idf,tfidf_sum
term_id,,,,,,,,,,,,,,,,,
3863,66,guerrilla,783,NN,0,0,guerrilla,65,0.080989,51678,50895,5.264274,0.001791,0.016346,9,0.845098,0.209746
1526,709,commerce,131,NN,0,0,commerc,367,0.013550,92879,48077,4.972797,0.000300,0.003508,15,0.623249,0.152544
9375,341,wages,254,NNS,0,0,wage,254,0.026272,86614,64516,6.673149,0.000581,0.006246,15,0.623249,0.151214
1147,110,capitalist,547,NN,0,0,capitalist,101,0.056578,60170,55247,5.714419,0.001251,0.012067,24,0.419129,0.149934
6592,145,price,470,NN,0,0,price,130,0.048614,68150,61100,6.319818,0.001075,0.010603,14,0.653213,0.133411
6658,81,production,698,NN,0,0,product,78,0.072197,56538,54444,5.631361,0.001597,0.014836,25,0.401401,0.121905
1145,125,capital,506,NN,0,0,capit,115,0.052338,63250,58190,6.018825,0.001158,0.011292,29,0.336943,0.114325
877,604,bernstein,153,NNP,0,0,bernstein,346,0.015825,92412,52938,5.475590,0.000350,0.004018,11,0.757948,0.113554
7697,4741,seller,3,NN,0,0,seller,495,0.000310,14223,1485,0.153600,0.000007,0.000118,1,1.799341,0.101849


In [50]:
VOCAB[['term_rank','term_str','pos_max','tfidf_sum']]\
    .sort_values('tfidf_sum', ascending=False).head(50)\
    .style.background_gradient(cmap=gradient_cmap, high=1)

,term_rank,term_str,pos_max,tfidf_sum
term_id,,,,
3863,66,guerrilla,NN,0.209746
1526,709,commerce,NN,0.152544
9375,341,wages,NNS,0.151214
1147,110,capitalist,NN,0.149934
6592,145,price,NN,0.133411
6658,81,production,NN,0.121905
1145,125,capital,NN,0.114325
877,604,bernstein,NNP,0.113554
7697,4741,seller,NN,0.101849


In [51]:
VOCAB.loc[VOCAB.pos_max != 'NNP', ['term_rank','term_str','pos_max','tfidf_sum']]\
    .sort_values('tfidf_sum', ascending=False)\
    .head(50).style.background_gradient(cmap=gradient_cmap, high=1)

,term_rank,term_str,pos_max,tfidf_sum
term_id,,,,
3863,66,guerrilla,NN,0.209746
1526,709,commerce,NN,0.152544
9375,341,wages,NNS,0.151214
1147,110,capitalist,NN,0.149934
6592,145,price,NN,0.133411
6658,81,production,NN,0.121905
1145,125,capital,NN,0.114325
7697,4741,seller,NN,0.101849
1937,226,cost,NN,0.098741


In [52]:
def tfidf_matrix(
    tokens: pd.DataFrame,
    bag: list,                        
    count_method: str = "n",           
    tf_method: str = "sum",            
    idf_method: str = "standard",      
) -> pd.DataFrame:
    bow = tokens.groupby(bag + ["term_id"]).size().rename("n").to_frame()
    bow["c"] = bow["n"].astype(bool).astype(int)

    if count_method == "binary":
        count_method = "c"
    if count_method not in ["n", "c"]:
        raise ValueError("count_method must be 'n', 'c', or 'binary'")

    dtcm = bow[count_method].unstack(fill_value=0)

    if tf_method == "sum":
        tf = dtcm.div(dtcm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    elif tf_method == "max":
        tf = dtcm.div(dtcm.max(axis=1).replace(0, np.nan), axis=0).fillna(0)
    elif tf_method == "log":
        tf = np.log10(1 + dtcm)
    elif tf_method == "raw":
        tf = dtcm.copy()
    elif tf_method == "double_norm":
        base = dtcm.div(dtcm.max(axis=1).replace(0, np.nan), axis=0).fillna(0)
        tf = base.copy()
        mask = tf > 0
        tf[mask] = tf_norm_k + (1 - tf_norm_k) * tf[mask]
    elif tf_method == "binary":
        tf = (dtcm > 0).astype(int)
    else:
        raise ValueError("tf_method must be one of: sum, max, log, double_norm, raw, binary")

    df = (dtcm > 0).sum(axis=0)
    N = dtcm.shape[0]

    if idf_method == "standard":
        idf = np.log10(N / df.replace(0, np.nan))
    elif idf_method == "max":
        idf = np.log10(df.max() / df.replace(0, np.nan))
    elif idf_method == "smooth":
        idf = np.log10((1 + N) / (1 + df)) + 1
    else:
        raise ValueError("idf_method must be one of: standard, max, smooth")

    idf = idf.replace([np.inf, -np.inf], 0).fillna(0)

    tfidf = tf.multiply(idf, axis=1)

    return tfidf

In [53]:
BOW['tf'] = TF.stack()
BOW['tfidf'] = TFIDF.stack()

In [54]:
TOKEN_F4 = TOK.merge(
    BOW.reset_index()[['book_id', 'chap_num', 'term_id', 'tf', 'tfidf']],
    on=['book_id', 'chap_num', 'term_id'],
    how='left'
).set_index(OHCO).sort_index()

In [55]:
VOCAB_F4 = VOCAB.copy()

VOCAB_F4['df'] = DF
VOCAB_F4['idf'] = IDF
VOCAB_F4['tfidf_sum'] = TFIDF.sum(axis=0)
VOCAB_F4['tfidf_mean'] = TFIDF.mean(axis=0)
VOCAB_F4['tfidf_max'] = TFIDF.max(axis=0)

In [56]:
f4_dir = Path("corpus/f4")
f4_dir.mkdir(parents=True, exist_ok=True)

VOCAB_F4.to_csv(f4_dir / "VOCAB.csv")
DTCM.to_csv(f4_dir / "DTCM.csv")
TFIDF.to_csv(f4_dir / "TFIDF.csv")